# CV3D Pipeline: Multi-Object Reconstruction, Masking & Nerfstudio Benchmark
**Project:** From Photos to 3D: NeRF and Gaussian Splatting in Practice  
**Authors:** TBN & Team  
**Runtime:** Google Colab (Pro recommended, T4/A100/L4 GPU)

This turnkey notebook executes the complete end-to-end photogrammetry and 3D reconstruction pipeline on local `/content/` NVMe storage using the self-contained archives prepared on Windows (`dataset_<OBJECT_ID>.tar` and `scripts.tar.gz`).

### Curated Objects
- `box_001`: Techcombank Gift Box (321 frames, SAM 2 dilated stool-subtraction masks)
- `basket_001`: Bamboo & Mesh Fruit Basket (362 frames, SAM 2 video predictor temporal masks)
- `figure_001`: Anime PVC Figurine (362 frames, clean draft masks)
- `stuff_toy_001`: Cotton Stuffed Horse (337 frames, SAM 2 video predictor temporal masks)

### Pipeline Stages
1. **Interactive Configuration**: Select target object and execution parameters via Colab form.
2. **Environment Preflight**: Check GPU and install COLMAP, FFmpeg, and Nerfstudio.
3. **Drive Mount & Dataset Extraction**: Transfer `.tar` archive to local runtime disk and verify checksums.
4. **Joint COLMAP SfM**: Run one unified reconstruction across all 5 trajectories (`orbit_`, `spiral_`, `blur_1_50_`, `blur_1_25_`, `holdout_`).
5. **Registration Auditing**: Audit registered vs. submitted image counts per capture condition.
6. **Mask Attachment**: Attach high-fidelity single-channel binary masks (0/255) to `transforms.json` with multi-scale downscaling (`masks/`, `masks_2/`, `masks_4/`).
7. **Condition Splits**: Deterministic farthest-point camera sampling for all condition groups against shared `holdout_` test set.
8. **Dataset Validation**: Rigorous schema, image, and mask dimension validation.
9. **500-Iteration Smoke Tests**: Validate `nerfacto` and `splatfacto` startup.
10. **Experiment Matrix & Metric Evaluation**: Benchmark PSNR, SSIM, LPIPS across conditions and view subsets.
11. **Archive & Cloud Sync**: Save trained checkpoints, evaluation logs, and processed SfM back to Google Drive.

In [ ]:
#@title ⚙️ Pipeline Configuration & Object Selection
#@markdown Select which captured object dataset to process and train in this Colab session.

OBJECT_ID = "box_001" #@param ["box_001", "basket_001", "figure_001", "stuff_toy_001"]
DRIVE_PROJECT_FOLDER = "CV3D_Final_Project" #@param {type:"string"}
DOWNSCALE_FACTOR = 2 #@param [1, 2, 4]
SFM_MATCHING_METHOD = "vocab_tree" #@param ["vocab_tree", "exhaustive"]
SMOKE_TEST_ONLY = True #@param {type:"boolean"}
FULL_TRAIN_ITERS = 30000 #@param {type:"integer"}

print(f"🎯 Target Object: {OBJECT_ID}")
print(f"📁 Google Drive Base: MyDrive/{DRIVE_PROJECT_FOLDER}")
print(f"🔍 Matching Method: {SFM_MATCHING_METHOD}")
print(f"🖼️ Downscale Factor: {DOWNSCALE_FACTOR}")
print(f"🚀 Mode: {'Smoke Test (500 iters)' if SMOKE_TEST_ONLY else f'Full Training ({FULL_TRAIN_ITERS} iters)'}")

In [ ]:
#@title 1. Environment & GPU Preflight
#@markdown Verifies GPU allocation and installs COLMAP, Nerfstudio, and dependencies.
!nvidia-smi
!apt-get update -qq
!apt-get install -y -qq colmap ffmpeg xvfb
!pip install -q --upgrade pip
!pip install -q nerfstudio rembg onnxruntime-gpu
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
!ns-process-data --help | head -n 5
!ns-train --help | head -n 5

In [ ]:
#@title 2. Mount Drive, Unpack Dataset & Restore Scripts
#@markdown Mounts Google Drive and copies the self-contained dataset archive to fast local runtime disk.
from google.colab import drive
from pathlib import Path
import shutil, os

# 1. Mount Google Drive
drive.mount('/content/drive')
DRIVE = Path(f'/content/drive/MyDrive/{DRIVE_PROJECT_FOLDER}')
WORK = Path('/content/project')
WORK.mkdir(parents=True, exist_ok=True)

# 2. Stage dataset archive on local NVMe disk
archive_filename = f'dataset_{OBJECT_ID}.tar'
drive_archive_path = DRIVE / '99_archives' / archive_filename
local_archive_path = Path(f'/content/{archive_filename}')

if drive_archive_path.exists():
    print(f'📥 Copying {archive_filename} from Google Drive...')
    shutil.copy2(drive_archive_path, local_archive_path)
    print(f'✅ Successfully copied archive ({local_archive_path.stat().st_size / (1024**3):.2f} GB)')
elif local_archive_path.exists():
    print(f'ℹ️ Found already uploaded archive at {local_archive_path}')
else:
    raise FileNotFoundError(f'Could not find {archive_filename} in {drive_archive_path} or /content/. Please verify upload to Google Drive.')

# 3. Extract dataset
data_root = WORK / 'data'
data_root.mkdir(parents=True, exist_ok=True)
print(f'📦 Extracting {archive_filename} to {data_root}...')
!tar -xf {local_archive_path} -C {data_root}

obj_dir = data_root / OBJECT_ID
img_count = len(list((obj_dir / 'images').glob('*.png')))
mask_count = len(list((obj_dir / 'masks').glob('*.png')))
print(f'📁 Verified object folder {obj_dir}: {img_count} images, {mask_count} masks')

# 4. Ensure images are standard 8-bit RGB (COLMAP compatibility)
import cv2
img_dir = obj_dir / 'images'
converted = 0
for p in img_dir.glob('*.png'):
    im = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if im is not None and im.dtype != 'uint8':
        cv2.imwrite(str(p), (im >> 8).astype('uint8'))
        converted += 1
if converted > 0:
    print(f'🔄 Converted {converted} images to 8-bit uint8.')
else:
    print('✅ All images are verified 8-bit uint8.')

# 5. Restore utility scripts
scripts_dir = WORK / 'scripts'
scripts_dir.mkdir(parents=True, exist_ok=True)

embedded_scripts = {'audit_registration.py': '"""Report COLMAP/Nerfstudio image registration by capture-condition prefix.\n\nExample:\n  python scripts/audit_registration.py \\\n      --input-images data/colmap_input/images \\\n      --transforms data/processed/joint_sfm/transforms.json \\\n      --output-csv data/manifests/registration_by_condition.csv\n\nThe input directory is the complete submitted image set. transforms.json contains\nonly images registered by COLMAP, so this computes registered/submitted counts.\n"""\nfrom __future__ import annotations\nimport argparse\nimport csv\nfrom pathlib import Path\n\nEXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}\nDEFAULT_PREFIXES = ("orbit_", "spiral_", "blur_1_50_", "blur_1_25_", "holdout_")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input-images", type=Path, required=True)\n    parser.add_argument("--transforms", type=Path, required=True)\n    parser.add_argument("--output-csv", type=Path, required=True)\n    parser.add_argument("--prefixes", default=",".join(DEFAULT_PREFIXES))\n    args = parser.parse_args()\n\n    if not args.input_images.is_dir():\n        raise SystemExit(f"Input image directory not found: {args.input_images}")\n    if not args.transforms.is_file():\n        raise SystemExit(f"Transforms JSON not found: {args.transforms}")\n\n    import json\n    meta = json.loads(args.transforms.read_text(encoding="utf-8"))\n    submitted = sorted(p.name for p in args.input_images.iterdir()\n                       if p.is_file() and p.suffix.lower() in EXTS)\n    registered = {Path(frame["file_path"]).name for frame in meta.get("frames", [])}\n    prefixes = [x.strip() for x in args.prefixes.split(",") if x.strip()]\n    if not submitted:\n        raise SystemExit("No supported image files found in input directory")\n\n    rows = []\n    total_registered = 0\n    for prefix in prefixes:\n        group_submitted = [n for n in submitted if n.startswith(prefix)]\n        group_registered = [n for n in group_submitted if n in registered]\n        n_sub, n_reg = len(group_submitted), len(group_registered)\n        total_registered += n_reg\n        fraction = n_reg / n_sub if n_sub else 0.0\n        rows.append({"condition_prefix": prefix, "submitted": n_sub,\n                     "registered": n_reg, "registration_fraction": f"{fraction:.6f}"})\n\n    ungrouped = [n for n in submitted if not any(n.startswith(p) for p in prefixes)]\n    if ungrouped:\n        rows.append({"condition_prefix": "UNMATCHED", "submitted": len(ungrouped),\n                     "registered": sum(n in registered for n in ungrouped),\n                     "registration_fraction": f"{sum(n in registered for n in ungrouped)/len(ungrouped):.6f}"})\n\n    args.output_csv.parent.mkdir(parents=True, exist_ok=True)\n    with args.output_csv.open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=["condition_prefix", "submitted", "registered", "registration_fraction"])\n        writer.writeheader()\n        writer.writerows(rows)\n\n    total_submitted = len(submitted)\n    total_registered = len(registered.intersection(submitted))\n    overall = total_registered / total_submitted\n    print(f"Overall registration: {total_registered}/{total_submitted} = {overall:.1%}")\n    for row in rows:\n        print(f"{row[\'condition_prefix\']}: {row[\'registered\']}/{row[\'submitted\']} = {float(row[\'registration_fraction\']):.1%}")\n    print(f"Wrote {args.output_csv}")\n\n\nif __name__ == "__main__":\n    main()\n', 'attach_masks.py': '"""Attach masks to a Nerfstudio transforms.json by exact filename stem.\n\nCopies/resizes masks to masks/, masks_2/, masks_4/ matching existing image scales.\nFails on missing matches rather than silently pairing by sorted index.\n"""\nfrom __future__ import annotations\nimport argparse, json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\n\ndef main() -> None:\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--dataset\',type=Path,required=True)\n    p.add_argument(\'--source-masks\',type=Path,required=True)\n    args=p.parse_args()\n    meta_path=args.dataset/\'transforms.json\'\n    if not meta_path.is_file(): raise SystemExit(f\'Missing {meta_path}\')\n    meta=json.loads(meta_path.read_text(encoding=\'utf-8\'))\n    frames=meta.get(\'frames\',[])\n    if not frames: raise SystemExit(\'transforms.json has no frames\')\n    masks={x.stem:x for x in args.source_masks.iterdir() if x.is_file() and x.suffix.lower()==\'.png\'}\n    missing=[]\n    for frame in frames:\n        image_rel=Path(frame[\'file_path\'])\n        image_path=args.dataset/image_rel\n        key=image_path.stem\n        if key not in masks: missing.append(key); continue\n        mask=cv2.imread(str(masks[key]),cv2.IMREAD_GRAYSCALE)\n        image=cv2.imread(str(image_path),cv2.IMREAD_COLOR)\n        if mask is None or image is None: raise SystemExit(f\'Unreadable image or mask for {key}\')\n        # Strict binary single-channel mask; black=ignore, white=keep.\n        mask=np.where(mask>=128,255,0).astype(np.uint8)\n        for scale,suffix in [(1,\'\'),(2,\'_2\'),(4,\'_4\')]:\n            image_dir=args.dataset/f\'images{suffix}\'\n            if not image_dir.is_dir(): continue\n            scaled_image=image if scale==1 else cv2.imread(str(image_dir/image_rel.name),cv2.IMREAD_COLOR)\n            if scaled_image is None: raise SystemExit(f\'Missing scaled image: {image_dir/image_rel.name}\')\n            h,w=scaled_image.shape[:2]\n            resized=cv2.resize(mask,(w,h),interpolation=cv2.INTER_NEAREST)\n            out_dir=args.dataset/f\'masks{suffix}\'; out_dir.mkdir(parents=True,exist_ok=True)\n            cv2.imwrite(str(out_dir/(image_rel.stem+\'.png\')),resized)\n        frame[\'mask_path\']=f\'masks/{image_rel.stem}.png\'\n    if missing:\n        raise SystemExit(\'No mask found for these image stems (no changes written): \'+\', \'.join(missing[:20]))\n    meta_path.write_text(json.dumps(meta,indent=2),encoding=\'utf-8\')\n    print(f\'Attached {len(frames)} masks by exact image stem.\')\n\nif __name__==\'__main__\': main()\n', 'create_condition_splits.py': '"""Create trajectory/blur and view-count splits with shared held-out cameras.\n\nInput must be one joint COLMAP/Nerfstudio reconstruction containing all image\nconditions. Prefixes are the filename stems used in the merged COLMAP input, e.g.\n  orbit_00001.png, spiral_00001.png, blur_1_50_00001.png, holdout_00001.png\n\nThe holdout prefix is split once into fixed validation/test sets. Each condition\ngets nested train subsets at requested ratios while using those same val/test\nimages and the same COLMAP coordinate system.\n"""\nfrom __future__ import annotations\nimport argparse, copy, json\nfrom pathlib import Path\nimport numpy as np\n\ndef fps(points: np.ndarray, count: int) -> list[int]:\n    if count<=0:return []\n    if count>=len(points):return list(range(len(points)))\n    first=int(np.argmax(np.sum((points-points.mean(axis=0))**2,axis=1)))\n    chosen=[first]; min_d=np.sum((points-points[first])**2,axis=1)\n    while len(chosen)<count:\n        j=int(np.argmax(min_d)); chosen.append(j)\n        min_d=np.minimum(min_d,np.sum((points-points[j])**2,axis=1))\n    return chosen\n\ndef main()->None:\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--dataset\',type=Path,required=True)\n    p.add_argument(\'--condition-prefix\',required=True,help=\'Filename stem prefix, e.g. orbit_\')\n    p.add_argument(\'--holdout-prefix\',default=\'holdout_\')\n    p.add_argument(\'--val-fraction\',type=float,default=0.35,help=\'Fraction of holdout pool reserved for validation; remainder is test.\')\n    p.add_argument(\'--ratios\',default=\'1.0,0.5,0.25,0.10\')\n    a=p.parse_args()\n    base=a.dataset/\'transforms.json\'; meta=json.loads(base.read_text(encoding=\'utf-8\'))\n    frames=meta.get(\'frames\',[])\n    if not frames:raise SystemExit(\'No frames in transforms.json\')\n    names=[]; poses=[]\n    for f in frames:\n        path=Path(f[\'file_path\']); names.append(f[\'file_path\']); poses.append(np.asarray(f[\'transform_matrix\'],dtype=float)[:3,3])\n    names_arr=np.asarray(names,dtype=object); poses_arr=np.asarray(poses)\n    hold=[i for i,n in enumerate(names) if Path(n).name.startswith(a.holdout_prefix)]\n    train=[i for i,n in enumerate(names) if Path(n).name.startswith(a.condition_prefix)]\n    if len(hold)<8:raise SystemExit(f\'Need >=8 registered holdout frames with prefix {a.holdout_prefix}; found {len(hold)}\')\n    if len(train)<10:raise SystemExit(f\'Need >=10 registered training frames with prefix {a.condition_prefix}; found {len(train)}\')\n    if not (0<a.val_fraction<0.8):raise SystemExit(\'--val-fraction must be between 0 and 0.8\')\n    test_n=max(2,min(len(hold)-2,round(len(hold)*(1.0-a.val_fraction))))\n    test_local=fps(poses_arr[hold],test_n); test=[hold[j] for j in test_local]\n    val_pool=[i for i in hold if i not in set(test)]\n    val_local=fps(poses_arr[val_pool],len(val_pool)); val=[val_pool[j] for j in val_local]\n    for r in [float(x) for x in a.ratios.split(\',\')]:\n        if not 0<r<=1:raise SystemExit(f\'Invalid ratio {r}\')\n        n=max(2,round(len(train)*r)); local=fps(poses_arr[train],min(n,len(train))); chosen=[train[j] for j in local]\n        out=copy.deepcopy(meta)\n        out[\'train_filenames\']=[names[i] for i in chosen]\n        out[\'val_filenames\']=[names[i] for i in val]\n        out[\'test_filenames\']=[names[i] for i in test]\n        out[\'split_protocol\']={\'condition_prefix\':a.condition_prefix,\'holdout_prefix\':a.holdout_prefix,\n          \'train_ratio\':r,\'train_count\':len(chosen),\'val_count\':len(val),\'test_count\':len(test),\n          \'shared_eval_cameras\':True,\'selection\':\'deterministic farthest-point sampling over COLMAP camera centres\'}\n        out_path=a.dataset/f\'transforms_{a.condition_prefix.rstrip("_")}_views_{int(round(r*100)):03d}.json\'\n        out_path.write_text(json.dumps(out,indent=2),encoding=\'utf-8\')\n        print(f\'{out_path.name}: train={len(chosen)}, val={len(val)}, test={len(test)}\')\n    print(\'Inspect the held-out views. This script uses camera-centre diversity, not a guarantee of perfect angular coverage.\')\n\nif __name__==\'__main__\':main()\n', 'validate_dataset.py': '"""Validate a Nerfstudio dataset JSON, image/mask paths, dimensions and splits."""\nfrom __future__ import annotations\nimport argparse, json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\n\ndef main() -> None:\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--dataset\',type=Path,required=True)\n    p.add_argument(\'--transforms\',default=\'transforms.json\')\n    p.add_argument(\'--require-masks\',action=\'store_true\')\n    args=p.parse_args()\n    meta_path=args.dataset/args.transforms\n    if not meta_path.is_file(): raise SystemExit(f\'Missing metadata: {meta_path}\')\n    meta=json.loads(meta_path.read_text(encoding=\'utf-8\'))\n    frames=meta.get(\'frames\',[])\n    errors=[]; warnings=[]\n    if not frames: errors.append(\'No frames in transforms JSON.\')\n    image_paths=[]; mask_count=0; fg_fracs=[]\n    for i,fr in enumerate(frames):\n        if \'file_path\' not in fr or \'transform_matrix\' not in fr:\n            errors.append(f\'Frame {i} lacks file_path or transform_matrix.\'); continue\n        image_path=args.dataset/fr[\'file_path\']\n        image=cv2.imread(str(image_path),cv2.IMREAD_COLOR)\n        if image is None:\n            errors.append(f\'Image missing/unreadable: {image_path}\'); continue\n        image_paths.append(fr[\'file_path\'])\n        mat=np.asarray(fr[\'transform_matrix\'],dtype=np.float64)\n        if mat.shape!=(4,4) or not np.isfinite(mat).all(): errors.append(f\'Invalid 4x4 pose: {fr["file_path"]}\')\n        if \'w\' in meta and int(meta[\'w\'])!=image.shape[1]: warnings.append(f\'Image width differs from top-level metadata: {fr["file_path"]}\')\n        if \'h\' in meta and int(meta[\'h\'])!=image.shape[0]: warnings.append(f\'Image height differs from top-level metadata: {fr["file_path"]}\')\n        mp=fr.get(\'mask_path\')\n        if args.require_masks and not mp: errors.append(f\'Missing mask_path: {fr["file_path"]}\')\n        if mp:\n            mask_path=args.dataset/mp\n            mask=cv2.imread(str(mask_path),cv2.IMREAD_UNCHANGED)\n            if mask is None: errors.append(f\'Mask missing/unreadable: {mask_path}\'); continue\n            if mask.ndim!=2: errors.append(f\'Mask must be one channel: {mask_path}\'); continue\n            if mask.shape!=image.shape[:2]: errors.append(f\'Mask/image size mismatch: {mask_path}\'); continue\n            vals=set(np.unique(mask).tolist())\n            if not vals.issubset({0,255}): errors.append(f\'Mask is not binary 0/255: {mask_path}\')\n            frac=float(np.mean(mask==255)); fg_fracs.append(frac); mask_count+=1\n            if frac<0.01 or frac>0.99: warnings.append(f\'Foreground fraction suspicious ({frac:.3f}): {mask_path}\')\n    name_set=set(image_paths)\n    for split in (\'train\',\'val\',\'test\'):\n        key=f\'{split}_filenames\'\n        if key in meta:\n            missing=set(meta[key])-name_set\n            if missing: errors.append(f\'{key} contains missing image paths: {sorted(missing)[:5]}\')\n    if all(k in meta for k in (\'train_filenames\',\'val_filenames\',\'test_filenames\')):\n        a=set(meta[\'train_filenames\']); b=set(meta[\'val_filenames\']); c=set(meta[\'test_filenames\'])\n        if a&b or a&c or b&c: errors.append(\'Train/val/test filename lists overlap.\')\n        if not c: errors.append(\'Test split is empty.\')\n        if not b: errors.append(\'Validation split is empty.\')\n    if args.require_masks and mask_count!=len(frames): errors.append(f\'Expected {len(frames)} masks, validated {mask_count}.\')\n    sparse=args.dataset/\'colmap\'/\'sparse\'/\'0\'\n    if not sparse.exists(): warnings.append(f\'COLMAP sparse model folder not found at {sparse}; inspect ns-process-data output layout.\')\n    print(f\'Frames declared: {len(frames)} | readable images: {len(image_paths)} | masks validated: {mask_count}\')\n    if fg_fracs: print(f\'Mask foreground fraction: median={np.median(fg_fracs):.3f}, min={np.min(fg_fracs):.3f}, max={np.max(fg_fracs):.3f}\')\n    for w in warnings: print(\'WARNING:\',w)\n    for e in errors: print(\'ERROR:\',e)\n    if errors: raise SystemExit(f\'VALIDATION FAILED ({len(errors)} errors, {len(warnings)} warnings)\')\n    print(f\'VALIDATION PASSED ({len(warnings)} warnings).\')\n\nif __name__==\'__main__\': main()\n', 'write_checksums.py': '"""Write a relative-path SHA-256 manifest for a directory tree."""\nfrom __future__ import annotations\nimport argparse, csv, hashlib\nfrom pathlib import Path\n\ndef digest(path:Path)->str:\n    h=hashlib.sha256()\n    with path.open(\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef main()->None:\n    p=argparse.ArgumentParser(); p.add_argument(\'--root\',type=Path,required=True); p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args(); files=sorted(x for x in a.root.rglob(\'*\') if x.is_file() and x.resolve()!=a.output.resolve())\n    a.output.parent.mkdir(parents=True,exist_ok=True)\n    with a.output.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        w=csv.writer(f); w.writerow([\'relative_path\',\'size_bytes\',\'sha256\'])\n        for x in files: w.writerow([x.relative_to(a.root).as_posix(),x.stat().st_size,digest(x)])\n    print(f\'Wrote checksums for {len(files)} files to {a.output}\')\n\nif __name__==\'__main__\': main()\n', 'verify_checksums.py': "from __future__ import annotations\nimport argparse,csv,hashlib\nfrom pathlib import Path\n\ndef digest(p:Path)->str:\n    h=hashlib.sha256()\n    with p.open('rb') as f:\n        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n    return h.hexdigest()\n\ndef main()->None:\n    ap=argparse.ArgumentParser(); ap.add_argument('--root',type=Path,required=True); ap.add_argument('--manifest',type=Path,required=True); a=ap.parse_args()\n    bad=[]; checked=0\n    with a.manifest.open(newline='',encoding='utf-8') as f:\n        for row in csv.DictReader(f):\n            p=a.root/row['relative_path']; checked+=1\n            if not p.is_file() or digest(p)!=row['sha256']: bad.append(row['relative_path'])\n    print(f'Checked {checked} files; failures={len(bad)}')\n    for x in bad[:30]: print('MISMATCH/MISSING:',x)\n    if bad: raise SystemExit(1)\n    print('All checksums match.')\nif __name__=='__main__':main()\n"}

for fname, code in embedded_scripts.items():
    (scripts_dir / fname).write_text(code, encoding='utf-8')
print(f'✅ Restored {len(embedded_scripts)} pipeline scripts in {scripts_dir.resolve()}')

In [ ]:
#@title 3. Joint COLMAP Reconstruction (Unified Coordinate System)
#@markdown Reconstructs all 5 trajectories in a single COLMAP run for joint camera calibration.
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

output_sfm_dir = f'/content/project/data/processed/{OBJECT_ID}/joint_sfm'

!export QT_QPA_PLATFORM=offscreen && ns-process-data images \
  --verbose \
  --data /content/project/data/{OBJECT_ID}/images \
  --output-dir {output_sfm_dir} \
  --camera-type perspective \
  --matching-method {SFM_MATCHING_METHOD} \
  --num-downscales 2 \
  --skip-image-processing \
  --no-gpu

In [ ]:
#@title 4. Audit Registration Rates by Capture Condition
#@markdown Analyzes COLMAP registration success across orbit, spiral, blur, and holdout passes.
import pandas as pd

audit_csv = f'/content/project/data/processed/{OBJECT_ID}/registration_by_condition.csv'

!python /content/project/scripts/audit_registration.py \
  --input-images /content/project/data/{OBJECT_ID}/images \
  --transforms {output_sfm_dir}/transforms.json \
  --output-csv {audit_csv}

df_audit = pd.read_csv(audit_csv)
print('\n📊 Registration Audit Summary:')
display(df_audit)

In [ ]:
#@title 5. Attach Pre-Computed High-Fidelity Masks to transforms.json
#@markdown Attaches SAM 2 / clean foreground masks with multi-scale downscaling matching image scales.
!python /content/project/scripts/attach_masks.py \
  --dataset {output_sfm_dir} \
  --source-masks /content/project/data/{OBJECT_ID}/masks

In [ ]:
#@title 6. Generate View-Count & Trajectory Splits
#@markdown Creates condition splits (100%, 50%, 25%, 10%) anchored to the shared held-out evaluation set.
conditions = ['orbit_', 'spiral_', 'blur_1_50_', 'blur_1_25_']

for cond in conditions:
    print(f'\n⚙️ Generating splits for condition: {cond}')
    !python /content/project/scripts/create_condition_splits.py \
      --dataset {output_sfm_dir} \
      --condition-prefix {cond} \
      --holdout-prefix holdout_ \
      --ratios 1.0,0.5,0.25,0.10

In [ ]:
#@title 7. Validate Dataset Schema & Mask Pairings
#@markdown Validates that all image/mask dimensions, channel depths, and split disjointness are verified.
!python /content/project/scripts/validate_dataset.py \
  --dataset {output_sfm_dir} \
  --transforms transforms_orbit_views_100.json \
  --require-masks

In [ ]:
#@title 8. 500-Iteration Smoke Tests (nerfacto & splatfacto)
#@markdown Verifies model initialization, CUDA loss backwards pass, and tensorboard logging.
print('🚀 Starting nerfacto smoke test (500 iters)...')
!ns-train nerfacto \
  --data {output_sfm_dir}/transforms_orbit_views_100.json \
  --max-num-iterations 500 \
  --vis tensorboard \
  nerfstudio-data --downscale-factor {DOWNSCALE_FACTOR}

print('\n🚀 Starting splatfacto smoke test (500 iters)...')
!ns-train splatfacto \
  --data {output_sfm_dir}/transforms_orbit_views_100.json \
  --max-num-iterations 500 \
  --vis tensorboard \
  nerfstudio-data --downscale-factor {DOWNSCALE_FACTOR}

In [ ]:
#@title 9. Run Experiment Matrix & Metric Evaluation
#@markdown Executes training and automated test-set evaluation (PSNR, SSIM, LPIPS).
import json, time
from pathlib import Path
import pandas as pd

if SMOKE_TEST_ONLY:
    print('ℹ️ SMOKE_TEST_ONLY is enabled. Skipping full 30,000-iteration training matrix.')
    print('👉 To run full benchmarks, uncheck SMOKE_TEST_ONLY in Cell 1 and re-run this cell.')
else:
    experiments = [
        ("nerfacto", "transforms_orbit_views_100.json", "orbit_100"),
        ("splatfacto", "transforms_orbit_views_100.json", "orbit_100"),
        ("nerfacto", "transforms_spiral_views_100.json", "spiral_100"),
        ("splatfacto", "transforms_spiral_views_100.json", "spiral_100"),
        ("nerfacto", "transforms_blur_1_50_views_100.json", "blur_1_50_100"),
        ("splatfacto", "transforms_blur_1_50_views_100.json", "blur_1_50_100"),
        ("nerfacto", "transforms_blur_1_25_views_100.json", "blur_1_25_100"),
        ("splatfacto", "transforms_blur_1_25_views_100.json", "blur_1_25_100"),
    ]
    
    results = []
    out_base = Path(f'/content/outputs/{OBJECT_ID}')
    out_base.mkdir(parents=True, exist_ok=True)
    
    for method, split_file, exp_name in experiments:
        split_path = f'{output_sfm_dir}/{split_file}'
        exp_dir = out_base / f'{exp_name}_{method}'
        print(f'\n==========================================')
        print(f'🔥 Training {method.upper()} on {exp_name}')
        print(f'==========================================')
        
        t0 = time.time()
        !ns-train {method} \
          --data {split_path} \
          --experiment-name {exp_name} \
          --output-dir {out_base} \
          --max-num-iterations {FULL_TRAIN_ITERS} \
          --vis tensorboard \
          nerfstudio-data --downscale-factor {DOWNSCALE_FACTOR}
        elapsed = time.time() - t0
        
        # Locate config.yml
        configs = list(exp_dir.glob('**/config.yml'))
        if configs:
            cfg_path = configs[0]
            eval_out = cfg_path.parent / 'test_metrics.json'
            print(f'📊 Evaluating test metrics on held-out views: {cfg_path}...')
            !ns-eval --load-config {cfg_path} --output-path {eval_out}
            
            if eval_out.exists():
                metrics = json.loads(eval_out.read_text(encoding='utf-8'))
                res_row = {
                    'object_id': OBJECT_ID,
                    'experiment': exp_name,
                    'method': method,
                    'wall_clock_sec': round(elapsed, 1),
                    'psnr': metrics.get('results', {}).get('psnr'),
                    'ssim': metrics.get('results', {}).get('ssim'),
                    'lpips': metrics.get('results', {}).get('lpips'),
                    'config_path': str(cfg_path)
                }
                results.append(res_row)
                print('✅ Result:', res_row)
                
    if results:
        df_res = pd.DataFrame(results)
        display(df_res)
        res_csv = f'/content/project/data/processed/{OBJECT_ID}/results.csv'
        df_res.to_csv(res_csv, index=False)

In [ ]:
#@title 10. Archive Processed Reconstruction & Sync to Google Drive
#@markdown Bundles joint_sfm (COLMAP poses, images, masks, splits) and results into Google Drive.
from pathlib import Path
import shutil

# 1. Write SHA-256 manifest of processed SfM
manifest_out = f'/content/project/data/processed/{OBJECT_ID}/processed_sha256.csv'
!python /content/project/scripts/write_checksums.py \
  --root {output_sfm_dir} \
  --output {manifest_out}

# 2. Package joint_sfm into tar
processed_archive = f'/content/processed_{OBJECT_ID}_joint_sfm.tar'
print(f'📦 Packaging processed SfM into {processed_archive}...')
!tar -cf {processed_archive} -C /content/project/data/processed/{OBJECT_ID} joint_sfm

# 3. Copy to Google Drive
dest_archive_dir = DRIVE / '99_archives'
dest_archive_dir.mkdir(parents=True, exist_ok=True)
dest_archive_path = dest_archive_dir / f'processed_{OBJECT_ID}_joint_sfm_v001.tar'

print(f'☁️ Syncing archive to Google Drive: {dest_archive_path}...')
shutil.copy2(processed_archive, dest_archive_path)

dest_doc_dir = DRIVE / '00_documentation' / 'manifests'
dest_doc_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(manifest_out, dest_doc_dir / f'processed_{OBJECT_ID}_sha256_v001.csv')

# Sync outputs/results if present
results_csv = Path(f'/content/project/data/processed/{OBJECT_ID}/results.csv')
if results_csv.exists():
    dest_res_dir = DRIVE / '07_final_results' / OBJECT_ID
    dest_res_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(results_csv, dest_res_dir / 'results.csv')

print('🎉 Complete! All deliverables and checkpoints safely synced to Google Drive.')